# nano-prover SFT on Colab (free T4 GPU)

Trains Qwen2.5-Coder on the graph corpus. **Prereq:** commit + push the repo first so this clones the latest.

In VS Code: open this file, click **Select Kernel → Colab**, sign in, pick a **GPU** runtime (Runtime → Change runtime type → T4 GPU). Then run top to bottom.

In [ ]:
# 1. confirm we have a GPU (must show a Tesla T4 / similar; if 'command not found' you're on CPU — fix the runtime)
!nvidia-smi

In [ ]:
# 2. clone (or update) the repo on the Colab machine.  Public repo: this just works.
#    Private repo: replace the URL with https://<GITHUB_TOKEN>@github.com/FELIXGAO0315/Conjecture-Gym.git
import os
if not os.path.isdir('Conjecture-Gym'):
    !git clone --depth 1 https://github.com/FELIXGAO0315/Conjecture-Gym.git
%cd Conjecture-Gym
!git pull
assert os.path.exists('output/train_corpus.jsonl'), 'corpus not found — did you push it?'

In [ ]:
# 3. install the SFT stack (Colab already has a CUDA torch)
!pip install -q -U transformers trl peft datasets accelerate

In [ ]:
# 4. build train/test splits from the corpus
!python training/prepare_data.py

In [ ]:
# 5. LoRA SFT.  ~tens of minutes on T4 for 1.5B / 3 epochs.
#    If you hit OOM: edit training/sft_train.py → BATCH=1, or switch MODEL to the 0.5B.
!python training/sft_train.py

## Get the adapter back (`training/out/`), then evaluate LOCALLY
Pick one:

In [ ]:
# OPTION A — zip for manual download (browser Colab: a download prompt; VS Code: find training/out.zip in the file tree)
!cd training && zip -rq out.zip out && ls -lh out.zip
try:
    from google.colab import files; files.download('training/out.zip')
except Exception as e:
    print('download via the file panel instead:', e)

In [ ]:
# OPTION B — push the adapter back to GitHub (needs a token with repo write).  Then `git pull` locally.
# TOKEN = 'ghp_xxx'   # a GitHub Personal Access Token (classic, repo scope)
# !git config user.email you@example.com && git config user.name you
# !git add -f training/out && git commit -m 'trained adapter'
# !git push https://{TOKEN}@github.com/FELIXGAO0315/Conjecture-Gym.git main

Back on your **local** machine (the Lean verifier lives there):
```bash
git pull                 # if you used Option B
# or unzip the downloaded out.zip into training/out/
python training/eval_lean.py --n 50 --k 1   # real Lean-verified pass@k
```